# NB02 — Demand Model (Dual-Rate Growth)
**Benin Least-Cost Electrification Analysis**

## What changed vs the original single-rate model

The original model applied a single 4%/yr growth rate to settlement demand.
This implicitly conflated two distinct phenomena that have different policy implications:

| Growth component |model | Rate |
|---|---|---|---|
| Per-HH intensity (income-driven) | Bundled in 4% | `demand_intensity_growth_rate` | 3%/yr |
| New households forming (population) | `population_growth_rate × rural_unelec_share` | 2.7% × 40% = 1.1%/yr |
| **Combined** | **4.0%** | **(1.03 × 1.011) − 1** | **≈ 4.1%** |


And **SHS price decline** is now computed: 5%/yr through 2030 → Tier-2 drops from $150 to $113 by 2030.

## Run order
Requires `data/processed/settlements_enriched.geojson` from NB00.


In [ ]:
import sys
sys.path.append('..')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.demand.demand_estimator import (
    add_demand_columns,
    shs_price_at_year,
    _intensity_factor,
    _hh_growth_factor,
    _tier_progression_factor,
)
from src.demand.mtf_tiers import assign_tiers_df
from src.config import GENERAL, DEMAND, SHS as SHS_CONFIG

sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')
print(f'  Demand intensity growth : {DEMAND["demand_intensity_growth_rate"]*100:.1f}%/yr (per-HH income)')
print(f'  Population growth rate  : {DEMAND["population_growth_rate"]*100:.1f}%/yr × '
      f'{DEMAND["rural_unelec_share"]*100:.0f}% rural share = '
      f'{DEMAND["population_growth_rate"]*DEMAND["rural_unelec_share"]*100:.2f}%/yr effective')
combined = ((1+DEMAND['demand_intensity_growth_rate'])*(1+DEMAND['population_growth_rate']*DEMAND['rural_unelec_share'])-1)*100
print(f'  Combined demand growth  : {combined:.2f}%/yr  (vs old single 4%)')
print(f'  Tier progression        : ramp-up over {DEMAND.get("tier_progression",{}).get("years_to_target_tier",5)} years')
shs_t2 = SHS_CONFIG['tier_2']['capex_per_unit']
print(f'  SHS Tier-2 price 2025   : ${shs_t2}  → ${shs_price_at_year(shs_t2, 2030):.0f} by 2030 (5%/yr decline)')


## 1. Load Data & Outlier Cap

Loads the GIS-enriched file from Notebook 00. Caps `energy_demand` at
the 99th percentile to remove implausible outliers before any calculation.


In [ ]:
from pathlib import Path

BASE_DIR      = Path('..').resolve()
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR       = BASE_DIR / 'data' / 'raw'

enriched = sorted(PROCESSED_DIR.glob('settlements_gis_enriched*.geojson'), reverse=True)
if enriched:
    gdf = gpd.read_file(enriched[0])
    print(f'Loaded enriched file: {enriched[0].name}')
else:
    gdf = gpd.read_file(RAW_DIR / 'Benin_settlement_properties.geojson')
    print('⚠ Enriched file not found — loaded raw VIDA')

print(f'Settlements : {len(gdf):,}  |  Columns: {len(gdf.columns)}')

# ── Outlier cap at p99 — must run before formula validation ──────────────
ed   = gdf['energy_demand']
p99  = ed.quantile(0.99)
n_out = (ed > p99).sum()
print(f'\nenergy_demand: median={ed.median():.1f}  mean={ed.mean():.1f}  '
      f'max={ed.max():.0f} kWh/day')
if ed.max() > 10_000:
    print(f'⚠ Capping {n_out} outliers at p99={p99:.0f} kWh/day')
    gdf['energy_demand'] = ed.clip(upper=p99)
    print(f'  New max = {gdf["energy_demand"].max():.1f} kWh/day ✓')


## 3. Electrification Status Calibration

**Run before the demand model** — `add_demand_columns()` uses `elec_status` to
apply different growth logic to electrified vs unelectrified settlements:
- Unelectrified: intensity growth + HH count growth + tier progression ramp-up
- Electrified: intensity growth only (densification handled separately in NB03)

Identifies electrified settlements using two proxy signals from VIDA/NB00:
- `NightLights` > 0.64 nW/cm²/sr (VIIRS 2020 — calibrated threshold)
- `GridDistKm` < 10 km (close to existing MV line)

Source: calibrated in the original NB02 against SBEE connection data.
The OR logic captures both grid-connected and well-lit peri-urban settlements.


In [ ]:
# ── Step 1: Check available columns ──────────────────────────────────────────
calib_cols = ['NightLights', 'GridDistKm', 'DistSubstation',
              'num_connections', 'num_buildings']
print('Calibration column check:')
for col in calib_cols:
    if col in gdf.columns:
        q = gdf[col].quantile([0.25,0.5,0.75,0.95]).round(3).to_dict()
        print(f'  ✓ {col:<22}: min={gdf[col].min():.3f}  '
              f'median={gdf[col].median():.3f}  '
              f'max={gdf[col].max():.3f}  '
              f'zeros={( gdf[col]==0).sum():,}')
    else:
        print(f'  ✗ {col:<22}: MISSING')

# ── Step 2: NightLights distribution ─────────────────────────────────────
if 'NightLights' in gdf.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('VIIRS Night-Time Lights — Calibration Basis', fontweight='bold')

    # Full distribution
    nl = gdf['NightLights']
    axes[0].hist(nl, bins=80, color='#1A237E', edgecolor='none')
    axes[0].set_xlabel('NightLights (nW/cm²/sr)')
    axes[0].set_ylabel('Settlements')
    axes[0].set_title('Full distribution')
    axes[0].set_yscale('log')

    # Zoomed 0–10 to see the threshold region
    axes[1].hist(nl.clip(upper=10), bins=80, color='#283593', edgecolor='none')
    axes[1].set_xlabel('NightLights (nW/cm²/sr) — clipped at 10')
    axes[1].set_title('Zoomed 0–10 (threshold region)')

    pct_zero = (nl == 0).sum() / len(nl) * 100
    print(f'\nNightLights = 0  : {(nl==0).sum():,} settlements ({pct_zero:.1f}%)')
    print(f'NightLights > 0  : {(nl>0).sum():,} settlements ({100-pct_zero:.1f}%)')
    print(f'NightLights > 1  : {(nl>1).sum():,} settlements ({(nl>1).sum()/len(nl)*100:.1f}%)')
    print(f'NightLights > 2  : {(nl>2).sum():,} settlements ({(nl>2).sum()/len(nl)*100:.1f}%)')
    print(f'NightLights > 5  : {(nl>5).sum():,} settlements ({(nl>5).sum()/len(nl)*100:.1f}%)')

    plt.tight_layout()
    plt.savefig('../data/outputs/maps/nightlights_distribution.png', dpi=150)
    plt.show()
else:
    print('⚠ NightLights column not found — run Notebook 00 first')


In [ ]:
# Calibrated thresholds — derived from sweep, locked in here
# NL > 0.64 OR GridDistKm < 10km → 7,882 settlements (45.8%) gap = +0.1 pp

CURRENT_ELEC_RATE = 0.457   # World Bank WDI 2023 — calibration anchor
NL_HIGH_THRESH    = 0.64    # nW/cm2/sr — above VIIRS noise floor
GRID_FULL_KM      = 10.0   # km — SBEE distribution reach
GRID_PARTIAL_KM   = 5.0    # km — ABERME MV spur limit
LV_RADIUS_KM      = 1.0    # km — LV densification radius
CONN_RATE_MIN     = 0.80   # connection rate threshold

# Confirm
nl   = gdf['NightLights']
gd   = gdf['GridDistKm']
mask = (nl > NL_HIGH_THRESH) | (gd < GRID_FULL_KM)
print(f'Thresholds set:')
print(f'  NL_HIGH_THRESH = {NL_HIGH_THRESH}  GRID_FULL_KM = {GRID_FULL_KM}')
print(f'  Electrified    = {mask.sum():,} ({mask.mean()*100:.1f}%)  '
      f'anchor {CURRENT_ELEC_RATE*100:.1f}%  gap {(mask.mean()-CURRENT_ELEC_RATE)*100:+.1f} pp')


In [ ]:
import numpy as np

nl = gdf['NightLights']
gd = gdf['GridDistKm']

# Binary classification — calibrated OR logic
# NL > 0.64 OR GridDistKm < 10km → electrified
gdf['elec_status'] = np.where(
    (nl > NL_HIGH_THRESH) | (gd < GRID_FULL_KM),
    'electrified',
    'unelectrified'
)

n_elec   = (gdf['elec_status'] == 'electrified').sum()
n_unelec = (gdf['elec_status'] == 'unelectrified').sum()
print('=== ELECTRIFICATION STATUS ===')
print(f'  electrified   : {n_elec:,} ({n_elec/len(gdf)*100:.1f}%)  → already served, grid anchor points')
print(f'  unelectrified : {n_unelec:,} ({n_unelec/len(gdf)*100:.1f}%)  → full least-cost analysis in NB03')
print(f'\n  Calibration anchor : {CURRENT_ELEC_RATE*100:.1f}%')
print(f'  Gap                : {(n_elec/len(gdf)-CURRENT_ELEC_RATE)*100:+.1f} pp  '
      f'({"✓ good" if abs(n_elec/len(gdf)-CURRENT_ELEC_RATE) < 0.02 else "⚠ check"})')
print(f'\nSanity check — median GridDistKm by status:')
print(gdf.groupby('elec_status')['GridDistKm'].median().round(1).to_string())
print(f'\nSanity check — median NightLights by status:')
print(gdf.groupby('elec_status')['NightLights'].median().round(3).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Electrification Calibration — Cross-Validation', fontsize=13, fontweight='bold')

colors = {'electrified': '#1565C0', 'unelectrified': '#EF9A9A'}

# 1. NightLights vs GridDistKm scatter
for status, grp in gdf.groupby('elec_status'):
    axes[0].scatter(grp['GridDistKm'].clip(upper=50),
                    grp['NightLights'].clip(upper=20),
                    s=2, alpha=0.3, color=colors[status], label=status)
axes[0].axhline(NL_HIGH_THRESH, color='black', lw=1, ls='--',
                label=f'NL threshold={NL_HIGH_THRESH}')
axes[0].axvline(GRID_FULL_KM, color='orange', lw=1, ls='--',
                label=f'Grid threshold={GRID_FULL_KM}km')
axes[0].set_xlabel('GridDistKm (clipped at 50km)')
axes[0].set_ylabel('NightLights (clipped at 20)')
axes[0].set_title('NightLights vs Grid Distance\n(OR logic — blue=electrified)')
axes[0].legend(fontsize=7, markerscale=3)

# 2. Bar chart
counts = gdf['elec_status'].value_counts()
bars = axes[1].bar(counts.index, counts.values,
                   color=[colors[s] for s in counts.index], edgecolor='white')
axes[1].set_title('Settlement Count by Status')
axes[1].set_ylabel('Settlements')
for bar, (status, v) in zip(bars, counts.items()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 f'{v:,}\n({v/len(gdf)*100:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/outputs/maps/electrification_calibration.png', dpi=150)
plt.show()


In [ ]:
# Gap analysis
n_elec   = (gdf['elec_status'] == 'electrified').sum()
n_unelec = (gdf['elec_status'] == 'unelectrified').sum()
N        = len(gdf)
rate     = n_elec / N
gap_pp   = (rate - CURRENT_ELEC_RATE) * 100

print('=== MODEL vs KNOWN CURRENT ELECTRIFICATION RATE ===')
print(f'  Known rate (World Bank WDI 2023) : {CURRENT_ELEC_RATE*100:.1f}%  ({int(CURRENT_ELEC_RATE*N):,} settlements)')
print(f'  Model rate                       : {rate*100:.1f}%  ({n_elec:,} settlements)')
print(f'  Gap                              : {gap_pp:+.1f} pp  '
      f'({"✓ good" if abs(gap_pp) < 2 else "⚠ acceptable" if abs(gap_pp) < 5 else "✗ poor"})')
print(f'\n  electrified   : {n_elec:,} ({rate*100:.1f}%)  → grid anchor points for NB03')
print(f'  unelectrified : {n_unelec:,} ({n_unelec/N*100:.1f}%)  → least-cost analysis')


---
## 4. Demand Model

`elec_status` is now set for all 17,205 settlements. Running demand estimation.


## 2. Validate VIDA Building Formula

VIDA computes `energy_demand` from satellite-detected building footprint sizes using fixed per-building
daily load weights:

```
energy_demand (kWh/day) = small_buildings  × 0.2170
                        + medium_buildings × 0.6500
                        + large_buildings  × 1.1000
```

This cell reverse-engineers the weights by running OLS regression of `energy_demand` on building counts.
A perfect fit (R² = 1.000, coefficients ≈ 0.217 / 0.650 / 1.100) confirms that VIDA's `energy_demand`
column is already the correct input for our demand model — **no recalculation needed**.

If the regression shows poor fit, it means the VIDA file version uses different weights and the
demand model should be re-calibrated against the actual coefficients.


In [ ]:
required = ['energy_demand','small_buildings','medium_buildings','large_buildings']
if all(c in gdf.columns for c in required):
    predicted = (gdf['small_buildings']  * 0.2170
               + gdf['medium_buildings'] * 0.6500
               + gdf['large_buildings']  * 1.1000)
    r   = gdf['energy_demand'].corr(predicted)
    mae = (gdf['energy_demand'] - predicted).abs().mean()
    print(f'VIDA formula validation (after p99 cap):')
    print(f'  r   = {r:.4f}  ')
    print(f'  MAE = {mae:.4f} kWh/day ')

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(gdf['energy_demand'], predicted, alpha=0.15, s=4,
               color='#1E88E5', rasterized=True)
    lim = gdf['energy_demand'].max() * 1.05
    ax.plot([0,lim],[0,lim],'r--',lw=1.5,label=f'Perfect fit (r={r:.4f})')
    ax.set_xlabel('VIDA energy_demand (kWh/day)')
    ax.set_ylabel('Formula prediction (kWh/day)')
    ax.set_title('VIDA Building Formula Validation')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../data/outputs/maps/vida_formula_validation.png', dpi=150)
    plt.show()
else:
    print('Building columns not available — skipping')


## 3. Run Demand Model (dual-rate)

`add_demand_columns()` now computes six new columns beyond the original five:

| Column | Formula | Example (100 HH, 3,650 kWh base) |
|---|---|---|
| `demand_year0_kwh` | VIDA base × tier_progression(0) | 730 kWh (Tier-1 at connection) |
| `demand_yearT_kwh` | base × intensity(15) × hh_growth(15) | 9,800 kWh |
| `num_households_yT` | HH × (1 + 1.1%)^15 | 118 HH |
| `hh_growth_total` | num_households_yT − num_households | 18 new HH |
| `shs_price_t2_y0` | $150 × (0.95)^0 | $150 |
| `shs_price_t2_yT` | $150 × (0.95)^5 floor at $105 | $113 |

The `demand_timeseries` list now reflects both growth components — values
grow faster in early years (tier progression ramp-up) then steadily after year 5.


In [ ]:
# ── Step 1: MTF tier assignment (needed for tier_progression in demand model) ──
gdf = assign_tiers_df(gdf)
print(f'MTF tier assignment ✓')
print(gdf['mtf_tier'].value_counts().sort_index().to_string())

# ── Step 2: Run demand model (dual growth rates + tier progression) ───────────
gdf = add_demand_columns(gdf)
print('\nDemand estimation complete ✓')
print(f'  vida_demand_year0_kwh  : {gdf["vida_demand_year0_kwh"].mean():.0f} mean kWh/yr')
print(f'  demand_year0_kwh       : {gdf["demand_year0_kwh"].mean():.0f} mean kWh/yr')
print(f'  demand_yearT_kwh       : {gdf["demand_yearT_kwh"].mean():.0f} mean kWh/yr')

# ── Step 3: Household count evolution ─────────────────────────────────────────
unelec = gdf[gdf['elec_status']=='unelectrified']
T = GENERAL['planning_horizon_years']
total_new_hh_growth = unelec['hh_growth_total'].sum()
print(f'\n  Unelectrified settlements : {len(unelec):,}')
print(f'  HH at t=0                 : {unelec["num_households"].sum():,.0f}')
print(f'  HH at t={T} (growth)       : {unelec["num_households_yT"].sum():,.0f}')
print(f'  New HH from pop. growth   : {total_new_hh_growth:,.0f} additional connections needed by {GENERAL["base_year"]+T}')
print(f'  SHS price 2025 / {GENERAL["base_year"]+T}  : ${gdf["shs_price_t2_y0"].iloc[0]:.0f} / ${gdf["shs_price_t2_yT"].iloc[0]:.0f}')


## 3b. Step-by-Step Verification — Single Settlement

Pick any settlement by index and trace the full demand calculation
manually. Cross-check each step against the VIDA DRE platform.


In [ ]:
# ── Dual-rate growth verification — single settlement trace ──────────────────
IDX = gdf[gdf['elec_status']=='unelectrified'].index[0]
row = gdf.loc[IDX]
T   = GENERAL['planning_horizon_years']

print('=' * 65)
print(f'SETTLEMENT: {row.get("village_name", row.get("name", f"index {IDX}"))}')
print(f'  elec_status   : {row["elec_status"]}')
print(f'  num_households: {row["num_households"]:.0f}')
print(f'  mtf_tier      : {row.get("mtf_tier", "N/A")}')
print(f'  energy_demand : {row.get("energy_demand",0):.3f} kWh/day  → {row["vida_demand_year0_kwh"]:.0f} kWh/yr (base)')
print('=' * 65)

print(f'\n{"Year":>4}  {"Intensity":>10}  {"HH Count":>8}  {"Tier Prog":>10}  {"Total kWh":>10}  {"HH (proj)":>10}')
print('-' * 65)
for t in [0, 1, 5, 10, 15]:
    i_f  = _intensity_factor(t)
    h_f  = _hh_growth_factor(t)
    tier_kwh = DEMAND['mtf_tiers'].get(f'tier_{row.get("mtf_tier",2)}', 73.0)
    t_f  = _tier_progression_factor(t, tier_kwh, is_unelectrified=True)
    d_t  = row['vida_demand_year0_kwh'] * i_f * h_f * t_f
    hh_t = row['num_households'] * h_f
    print(f'{t:>4}  {i_f:>10.4f}  {h_f:>8.4f}  {t_f:>10.4f}  {d_t:>10,.0f}  {hh_t:>10.1f}')

print(f'\n  Net combined growth rate: {((row["demand_yearT_kwh"]/row["demand_year0_kwh"])**(1/T)-1)*100:.2f}%/yr')
print(f'  Old single-rate (4%):     {(1.04**T - 1)*100:.0f}% total growth')
print(f'  New dual-rate:            {(row["demand_yearT_kwh"]/row["demand_year0_kwh"] - 1)*100:.0f}% total growth')


In [ ]:
# ── Dual-rate growth verification ─────────────────────────────────────────────
# Pick one unelectrified settlement and trace through both growth components.
# Tries to find a Tier-2+ settlement so tier progression ramp is visible.

unelec = gdf[gdf["elec_status"]=="unelectrified"]
# Prefer a Tier-2+ settlement so the progression ramp is visible
for idx in unelec.index:
    if unelec.loc[idx, "mtf_tier"] >= 2:
        IDX = idx
        break
else:
    IDX = unelec.index[0]

row = gdf.loc[IDX]

from src.demand.demand_estimator import (
    _intensity_factor, _hh_growth_factor, _tier_progression_factor
)
from src.config import DEMAND, GENERAL

T = GENERAL["planning_horizon_years"]
base_hh  = float(row.get("num_connections", 1) or 1)
base_dem = float(row.get("vida_demand_year0_kwh", 0))
tier     = int(row.get("mtf_tier", 2))
tier_kwh = DEMAND["mtf_tiers"].get(f"tier_{tier}", 73.0)
tier1_kwh= DEMAND["mtf_tiers"]["tier_1"]

print(f"Settlement index    : {IDX}")
print(f"Base HH (t=0)       : {base_hh:.0f}")
print(f"Base demand (t=0)   : {base_dem:,.0f} kWh/yr")
print(f"Assigned MTF tier   : Tier-{tier} ({tier_kwh:.0f} kWh/HH/yr)")
if tier_kwh <= tier1_kwh:
    print("  → Tier-1: no progression ramp (already at base tier)")
print()
print(f"{{'Year':>4}}  {{'Intensity':>10}}  {{'HH growth':>10}}  {{'Tier prog':>10}}  {{'Total HH':>9}}  {{'Demand kWh':>12}}")
print("-"*65)
for t in [0, 1, 2, 3, 5, 8, 10, 15]:
    intf = _intensity_factor(t)
    hhf  = _hh_growth_factor(t)
    tpf  = _tier_progression_factor(t, tier_kwh, is_unelectrified=True)
    hh_t = base_hh * hhf
    dem_t = base_dem * intf * hhf * tpf
    print(f"{t:>4}  {intf:>10.4f}  {hhf:>10.4f}  {tpf:>10.4f}  {hh_t:>9.1f}  {dem_t:>12,.0f}")

print()
print(f"Growth ratio Y0→Y15 : {gdf.loc[IDX,'demand_yearT_kwh']/max(gdf.loc[IDX,'demand_year0_kwh'],1):.2f}x")
print(f"New HH by 2040      : {gdf.loc[IDX,'hh_growth_total']:.0f}")
print(f"SHS Tier-2 price now: ${gdf.loc[IDX,'shs_price_t2_y0']:.0f}  →  ${gdf.loc[IDX,'shs_price_t2_yT']:.0f} by {GENERAL['base_year']+T}")


## 4. Productive Use Uplift

Scales `demand_year0_kwh` by `ProductiveUseFactor` from MODIS land cover
(Notebook 00): ×1.25 cropland, ×1.40 urban, ×1.00 baseline.


In [ ]:
if 'ProductiveUseFactor' in gdf.columns:
    gdf['demand_year0_kwh']      = gdf['demand_year0_kwh']      * gdf['ProductiveUseFactor']
    gdf['demand_yearT_kwh']      = gdf['demand_yearT_kwh']      * gdf['ProductiveUseFactor']
    gdf['vida_demand_year0_kwh'] = gdf['vida_demand_year0_kwh'] * gdf['ProductiveUseFactor']
    uplifted = (gdf['ProductiveUseFactor'] > 1.0).sum()
    print(f'ProductiveUseFactor applied:')
    print(gdf['ProductiveUseFactor'].value_counts().sort_index().to_string())
    print(f'  {uplifted:,} settlements uplifted ({uplifted/len(gdf)*100:.1f}%)')
else:
    print('⚠ ProductiveUseFactor not found — no uplift applied')


## 5. Summary Statistics (dual-rate model)

Key numbers to check after running with dual-rate growth:

- **Intensity growth multiplier** at year 15: `1.03^15 = 1.56×` (per-HH demand grows 56%)
- **HH growth multiplier** at year 15: `1.011^15 = 1.18×` (18% more households by 2040)
- **Combined multiplier**: `1.56 × 1.18 = 1.84×` (vs old model's `1.04^15 = 1.80×`)
- **New HH total** (unelectrified settlements): should be ~120,000–180,000 additional HH by 2040
- **Year-0 demand with tier progression**: lower than old model (HH start at Tier-1)
- **Year-15 demand**: slightly higher than old model (HH count grows)

The SHS price decline columns confirm:
- Tier-2 kit: $150 (2025) → ~$113 (2030 floor) — reduces SHS LCOE in later phases


In [ ]:
T    = GENERAL['planning_horizon_years']
base = GENERAL['base_year']

unelec = gdf[gdf['elec_status']=='unelectrified']
elec   = gdf[gdf['elec_status']=='electrified']

print('=== DEMAND MODEL SUMMARY ===')
print(f'  Settlements total       : {len(gdf):,}')
print(f'  Unelectrified           : {len(unelec):,}')
print(f'  Planning horizon        : {T} years ({base}–{base+T})')
print()
print('  DEMAND:')
print(f'  Unelec demand Year 0    : {unelec["demand_year0_kwh"].sum()/1e6:.1f} GWh/yr')
print(f'  Unelec demand Year {T}   : {unelec["demand_yearT_kwh"].sum()/1e6:.1f} GWh/yr')
combined = ((1+DEMAND['demand_intensity_growth_rate'])*(1+DEMAND['population_growth_rate']*DEMAND['rural_unelec_share'])-1)*100
print(f'  Combined growth rate    : {combined:.2f}%/yr (intensity {DEMAND["demand_intensity_growth_rate"]*100:.0f}% + HH {DEMAND["population_growth_rate"]*DEMAND["rural_unelec_share"]*100:.2f}%)')
print()
print('  HOUSEHOLDS:')
print(f'  Unelec HH at t=0        : {unelec["num_households"].sum():,.0f}')
print(f'  Unelec HH at t={T}       : {unelec["num_households_yT"].sum():,.0f}')
print(f'  New HH from pop growth  : {unelec["hh_growth_total"].sum():,.0f}  (need first-time connection)')
print()
print('  SHS PRICE DECLINE:')
print(f'  Tier-2 kit {base}       : ${gdf["shs_price_t2_y0"].iloc[0]:.0f}')
print(f'  Tier-2 kit {base+T}      : ${gdf["shs_price_t2_yT"].iloc[0]:.0f}  (5%/yr → floor at 70%)')


## 6. Visualisation

Three-panel chart for the demand model outputs:

1. **Left** — histogram of `demand_year0_kwh` (log scale) for unelectrified settlements.
   Expected shape: right-skewed, peak at ~1,000–3,000 kWh/yr.

2. **Centre** — scatter of `demand_year0_kwh` vs `num_connections` coloured by MTF tier.
   Expected: near-linear, with Tier-3/4 settlements above the main trend.

3. **Right** — demand growth curves for a sample of settlements showing the 4%/yr compound growth.
   The spread should widen over time as absolute differences grow with the base.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('VIDA Demand Model — Benin 17,205 Settlements', fontsize=13, fontweight='bold')

# 1. Demand distribution
d = gdf['demand_year0_kwh'].clip(upper=gdf['demand_year0_kwh'].quantile(0.95))
axes[0].hist(d, bins=60, color='#2196F3', edgecolor='white')
axes[0].set_title('Demand Distribution\n(Year 0, clipped at p95 for display)')
axes[0].set_xlabel('kWh/year')
axes[0].set_ylabel('Settlements')

# 2. VIDA base vs total (incl. institutional)
axes[1].scatter(gdf['vida_demand_year0_kwh'], gdf['demand_year0_kwh'],
                alpha=0.2, s=4, color='#4CAF50')
lim = gdf['demand_year0_kwh'].quantile(0.95)
axes[1].plot([0,lim],[0,lim],'r--',alpha=0.5,label='No institutional')
axes[1].set_title('VIDA Base vs Total Demand\n(incl. institutional load)')
axes[1].set_xlabel('VIDA base (kWh/year)')
axes[1].set_ylabel('Total demand (kWh/year)')
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, lim); axes[1].set_ylim(0, lim)

# 3. Year 0 vs Year T growth
axes[2].scatter(gdf['demand_year0_kwh'], gdf['demand_yearT_kwh'],
                alpha=0.2, s=4, color='#9C27B0')
lim2 = gdf['demand_yearT_kwh'].quantile(0.95)
axes[2].plot([0,lim2],[0,lim2],'r--',alpha=0.5,label='No growth')
T = GENERAL['planning_horizon_years']
g = DEMAND['demand_growth_rate']
axes[2].set_title(f'Year 0 → Year {T} Growth\n({g*100:.0f}%/year compound)')
axes[2].set_xlabel('Year 0 (kWh/year)')
axes[2].set_ylabel(f'Year {T} (kWh/year)')
axes[2].legend(fontsize=8)
axes[2].set_xlim(0, lim2); axes[2].set_ylim(0, lim2)

plt.tight_layout()
plt.savefig('../data/outputs/maps/demand_results.png', dpi=150)
plt.show()


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.ops import unary_union

colors = {'electrified': '#1565C0', 'unelectrified': '#EF9A9A'}
sizes  = {'electrified': 4, 'unelectrified': 1}

fig, ax = plt.subplots(figsize=(8, 11))
ax.set_facecolor('#E8F4F8')

# ── Country boundary from admin file or convex hull of settlements ────────────
boundary_drawn = False
# Try admin boundary file first
for admin_path in [
    Path('..') / 'data' / 'raw' / 'Benin_boundary.gpkg',
    Path('..') / 'data' / 'raw' / 'benin_boundary.gpkg',
    Path('..') / 'data' / 'raw' / 'benin_admin0.geojson',
]:
    if admin_path.exists():
        import geopandas as gpd
        admin = gpd.read_file(admin_path)
        admin.plot(ax=ax, color='white', edgecolor='#333333', linewidth=1.2, zorder=1)
        boundary_drawn = True
        print(f'Boundary from: {admin_path.name}')
        break

if not boundary_drawn:
    # Derive boundary from convex hull of all settlements
    import geopandas as gpd
    from shapely.ops import unary_union
    hull = gpd.GeoDataFrame(
        geometry=[unary_union(gdf.geometry).convex_hull],
        crs=gdf.crs
    )
    hull.plot(ax=ax, color='white', edgecolor='#333333', linewidth=1.2, zorder=1)
    print('Boundary: convex hull of settlements (no admin file found)')

# ── Transmission lines ───────────────────────────────────────────────────────
TRANS_PATH = Path('..') / 'data' / 'raw' / 'Benin_existing_transmission_lines_2017.geojson'
if TRANS_PATH.exists():
    lines = gpd.read_file(TRANS_PATH)
    lines.plot(ax=ax, color='#212121', linewidth=0.8, alpha=0.7, zorder=2)

# ── Settlements ──────────────────────────────────────────────────────────────
for status in ['unelectrified', 'electrified']:
    sub = gdf[gdf['elec_status'] == status]
    if len(sub) == 0: continue
    sub.plot(ax=ax, color=colors[status],
             markersize=sizes[status], alpha=0.7,
             marker='o', linewidth=0, zorder=3)

# ── Extent ───────────────────────────────────────────────────────────────────
ax.set_xlim(0.8, 3.9)
ax.set_ylim(6.1, 12.5)

# ── Legend ───────────────────────────────────────────────────────────────────
n_elec   = (gdf['elec_status'] == 'electrified').sum()
n_unelec = (gdf['elec_status'] == 'unelectrified').sum()
legend_elements = [
    mpatches.Patch(color='#1565C0', label=f'Electrified ({n_elec:,} — {n_elec/len(gdf)*100:.1f}%)'),
    mpatches.Patch(color='#EF9A9A', label=f'Unelectrified ({n_unelec:,} — {n_unelec/len(gdf)*100:.1f}%)'),
    mpatches.Patch(color='#212121', label='Transmission lines'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9,
          framealpha=0.95, edgecolor='gray')
ax.set_title(
    f'Benin — Electrification Status ({len(gdf):,} settlements)\n'
    f'NL > {NL_HIGH_THRESH} OR GridDist < {GRID_FULL_KM}km  |  '
    f'Electrified: {n_elec/len(gdf)*100:.1f}%  anchor: {CURRENT_ELEC_RATE*100:.1f}%',
    fontsize=11, fontweight='bold'
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig('../data/outputs/maps/electrification_status_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Map saved ✓')


In [ ]:
import folium
from folium.plugins import FastMarkerCluster

# ── Interactive Folium map — electrification status ───────────────────────────
m = folium.Map(
    location=[9.3, 2.3],   # Benin centre
    zoom_start=7,
    tiles='CartoDB positron'
)

colors = {'electrified': '#1565C0', 'unelectrified': '#E53935'}
radii  = {'electrified': 4, 'unelectrified': 2}

# Add settlements by status
for status in ['unelectrified', 'electrified']:
    sub = gdf[gdf['elec_status'] == status].copy()
    sub['lon'] = sub.geometry.centroid.x
    sub['lat'] = sub.geometry.centroid.y
    color = colors[status]
    r     = radii[status]
    for _, row in sub.iterrows():
        name = row.get('village_name', row.get('name', 'N/A'))
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=r,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            weight=0,
            popup=folium.Popup(
                f'<b>{name}</b><br>'
                f'Status: {status}<br>'
                f'NightLights: {row.get("NightLights", 0):.3f}<br>'
                f'GridDistKm: {row.get("GridDistKm", 0):.1f} km<br>'
                f'Demand Y0: {row.get("demand_year0_kwh", 0):,.0f} kWh/yr',
                max_width=220
            )
        ).add_to(m)

# Transmission lines
TRANS_PATH = Path('..') / 'data' / 'raw' / 'Benin_existing_transmission_lines_2017.geojson'
if TRANS_PATH.exists():
    folium.GeoJson(
        str(TRANS_PATH),
        name='Transmission lines',
        style_function=lambda x: {
            'color': '#212121', 'weight': 1.5, 'opacity': 0.8
        }
    ).add_to(m)

# Legend
n_elec   = (gdf['elec_status'] == 'electrified').sum()
n_unelec = (gdf['elec_status'] == 'unelectrified').sum()
legend_html = f'''
<div style="position:fixed; bottom:40px; left:40px; z-index:1000;
            background:white; padding:12px 16px; border-radius:8px;
            border:1px solid #ccc; font-size:13px; line-height:1.8;">
  <b>Electrification Status</b><br>
  <span style="color:#1565C0">&#9679;</span> Electrified &nbsp;&nbsp;{n_elec:,} ({n_elec/len(gdf)*100:.1f}%)<br>
  <span style="color:#E53935">&#9679;</span> Unelectrified &nbsp;{n_unelec:,} ({n_unelec/len(gdf)*100:.1f}%)<br>
  <hr style="margin:6px 0">
  <small>NL &gt; {NL_HIGH_THRESH} OR GridDist &lt; {GRID_FULL_KM}km<br>
  Anchor: {CURRENT_ELEC_RATE*100:.1f}% (World Bank WDI 2023)</small>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)

OUT_MAP = Path('..') / 'data' / 'outputs' / 'maps'
OUT_MAP.mkdir(parents=True, exist_ok=True)
m.save(str(OUT_MAP / 'electrification_status_map.html'))
print('Interactive map saved → data/outputs/maps/electrification_status_map.html')
m


In [ ]:
from pathlib import Path
from datetime import datetime
import json as _json

OUT_DIR   = Path('..') / 'data' / 'processed'
TABLE_DIR = Path('..') / 'data' / 'outputs' / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M')

drop_cols = [c for c in gdf.columns if 'institutional' in c.lower()]
if drop_cols:
    gdf = gdf.drop(columns=drop_cols)

# Full dataset — all 17,205 settlements with elec_status
# NB03 filters internally: gdf[gdf['elec_status'] == 'unelectrified']
full_path = OUT_DIR / f'settlements_demand_{ts}.geojson'
gdf.to_file(full_path, driver='GeoJSON')
print(f'Saved : {full_path.name}  ({len(gdf):,} settlements)')
print(f'  electrified   : {(gdf.elec_status=="electrified").sum():,}')
print(f'  unelectrified : {(gdf.elec_status=="unelectrified").sum():,}')

# Calibration params
params = {
    'timestamp'         : ts,
    'current_elec_rate' : CURRENT_ELEC_RATE,
    'nl_high_threshold' : NL_HIGH_THRESH,
    'grid_full_km'      : GRID_FULL_KM,
    'calibration_logic' : f'NightLights > {NL_HIGH_THRESH} OR GridDistKm < {GRID_FULL_KM}km',
    'source_nightlights': 'VIIRS DNB 2020 (NASA/NOAA)',
    'source_grid_dist'  : 'VIDA distance_to_existing_transmission_lines',
    'reference_rate'    : 'World Bank WDI 2023 — Benin 45.7%',
}
with open(TABLE_DIR / f'calibration_params_{ts}.json', 'w') as f:
    _json.dump(params, f, indent=2)
print(f'Calibration params  : calibration_params_{ts}.json')
print(f'\nNB02 complete ✓  gap = +0.1 pp — ready for NB03')
